In [1]:
import wrds
db = wrds.Connection(wrds_username='liaojy')

Loading library list...
Done


In [2]:
db.describe_table(library='crsp', table='msf')

Approximately 5153763 rows in crsp.msf.


,name,nullable,type,comment
0,cusip,True,VARCHAR(8),CUSIP Header
1,permno,True,INTEGER,PERMNO
2,permco,True,INTEGER,PERMCO
3,issuno,True,INTEGER,Nasdaq Issue Number
4,hexcd,True,SMALLINT,Exchange Code Header
5,hsiccd,True,INTEGER,Standard Industrial Classification Code Header
6,date,True,DATE,Date of Observation
7,bidlo,True,"NUMERIC(11, 5)",Bid or Low Price
8,askhi,True,"NUMERIC(11, 5)",Ask or High Price
9,prc,True,"NUMERIC(11, 5)",Price or Bid/Ask Average


In [3]:
query = """
SELECT permno, date, ret, prc, shrout
FROM crsp.msf
WHERE date >= '2020-01-01' AND date <= '2024-12-31'
"""
df = db.raw_sql(query, date_cols=['date'])
df = df.sort_values(['permno','date']).reset_index(drop=True)

print(df.shape)
print(df.head())

(546882, 5)
   permno       date       ret        prc   shrout
0   10026 2020-01-31 -0.100016     165.84  18919.0
1   10026 2020-02-28  -0.03027  160.82001  18919.0
2   10026 2020-03-31 -0.244031      121.0  18888.0
3   10026 2020-04-30  0.049835     127.03  18888.0
4   10026 2020-05-29  0.012596     128.63  18888.0


In [4]:
import numpy as np
df['logret'] = np.log(1+df['ret'])
df['mom_12_2'] = df.groupby('permno')['logret'].transform(lambda x: x.shift(1).rolling(window=11).sum())
df['mktcap'] = df['prc'].abs() * df['shrout']
print(df[['permno','date','ret','mom_12_2', 'mktcap']].head(20))


    permno       date       ret  mom_12_2         mktcap
0    10026 2020-01-31 -0.100016       NaN     3137526.96
1    10026 2020-02-28  -0.03027       NaN  3042553.76919
2    10026 2020-03-31 -0.244031       NaN      2285448.0
3    10026 2020-04-30  0.049835       NaN     2399342.64
4    10026 2020-05-29  0.012596       NaN     2429563.44
5    10026 2020-06-30 -0.007191       NaN     2401231.44
6    10026 2020-07-31 -0.031464       NaN     2326541.35
7    10026 2020-08-31  0.104118       NaN     2568775.25
8    10026 2020-09-30 -0.036668       NaN     2466326.85
9    10026 2020-10-30  0.039727       NaN  2564306.73915
10   10026 2020-11-30  0.072435       NaN     2755722.06
11   10026 2020-12-31  0.072598 -0.223327     2945193.72
12   10026 2021-01-29 -0.017442 -0.047865      2897486.8
13   10026 2021-02-26  0.039958 -0.034724   3013264.6102
14   10026 2021-03-31 -0.007275  0.284211     2988909.02
15   10026 2021-04-30  0.048271  0.228277     3133515.96
16   10026 2021-05-28  0.066642

In [5]:
db.describe_table(library='comp',table='funda')

Approximately 941436 rows in comp.funda.


,name,nullable,type,comment
0,gvkey,True,VARCHAR(7),Global Company Key
1,datadate,True,DATE,Data Date
2,fyear,True,INTEGER,Data Year - Fiscal
3,indfmt,True,VARCHAR(13),Industry Format
4,consol,True,VARCHAR(3),Level of Consolidation - Company Annual Descri...
...,...,...,...,...
944,au,True,VARCHAR(9),Auditor
945,auop,True,VARCHAR(9),Auditor Opinion
946,auopic,True,VARCHAR(2),Auditor Opinion - Internal Control
947,ceoso,True,VARCHAR(2),Chief Executive Officer SOX Certification


In [6]:
query = """
    SELECT gvkey, datadate, fyear, ceq, pstk, txditc, seq, at, lt
    FROM comp.funda
    WHERE datadate >= '2020-01-01' AND datadate <= '2024-12-31'
    AND indfmt = 'INDL'
    AND datafmt = 'STD'
    AND popsrc = 'D'
    AND consol = 'C'
"""

comp = db.raw_sql(query, date_cols=['datadate'])

print(comp.head(15))
print("shape:", comp.shape)

     gvkey   datadate  fyear      ceq  pstk  txditc      seq       at       lt
0   001004 2020-05-31   2019    902.6   0.0     0.0    902.6   2079.0   1176.4
1   001004 2021-05-31   2020    974.4   0.0     9.5    974.4   1539.7    565.3
2   001004 2022-05-31   2021   1034.5   0.0    20.0   1034.5   1573.9    539.4
3   001004 2023-05-31   2022   1099.1   0.0    33.6   1099.1   1833.1    734.0
4   001004 2024-05-31   2023   1189.8   0.0    23.9   1189.8   2770.0   1580.2
5   001019 2020-12-31   2020   13.479   0.0   0.361   13.479    40.57   27.091
6   001045 2020-12-31   2020  -6867.0   0.0     9.0  -6867.0  62008.0  68875.0
7   001045 2021-12-31   2021  -7340.0   0.0     9.0  -7340.0  66467.0  73807.0
8   001045 2022-12-31   2022  -5799.0   0.0    10.0  -5799.0  64716.0  70515.0
9   001045 2023-12-31   2023  -5202.0   0.0     9.0  -5202.0  63058.0  68260.0
10  001045 2024-12-31   2024  -3977.0   0.0     9.0  -3977.0  61783.0  65760.0
11  001050 2020-12-31   2020  202.658   0.0    6.97 

In [7]:
link_query = """
    SELECT gvkey, lpermno AS permno, linktype, linkprim, linkdt, linkenddt
    FROM crsp.ccmxpf_linktable
    WHERE linktype IN ('LU', 'LC')
    AND linkprim IN ('P', 'C')
"""
link = db.raw_sql(link_query, date_cols=['linkdt', 'linkenddt'])

print(link.head(15))
print("shape:", link.shape)

     gvkey   permno linktype linkprim     linkdt  linkenddt
0   001000  25881.0       LU        P 1970-11-13 1978-06-30
1   001001  10015.0       LU        P 1983-09-20 1986-07-31
2   001002  10023.0       LC        C 1972-12-14 1973-06-05
3   001003  10031.0       LU        C 1983-12-07 1989-08-16
4   001004  54594.0       LU        P 1972-04-24        NaT
5   001005  61903.0       LU        C 1973-01-31 1983-01-31
6   001007  10058.0       LU        C 1973-10-01 1979-01-30
7   001007  10058.0       LU        P 1979-01-31 1984-09-28
8   001008  10066.0       LC        P 1983-08-25 1987-02-26
9   001009  10074.0       LC        C 1982-01-18 1996-03-13
10  001010  10006.0       LU        C 1950-05-01 1962-01-30
11  001010  10006.0       LU        P 1962-01-31 1984-06-28
12  001011  10082.0       LC        P 1983-03-21 1995-09-28
13  001012  10103.0       LU        P 1978-01-31 1989-12-29
14  001013  50906.0       LU        P 1979-03-16 2010-12-31
shape: (33324, 6)


In [8]:
comp_linked = comp.merge(link, on = 'gvkey', how = 'inner')

print(comp_linked.shape)
print(comp_linked[['gvkey','datadate','permno','linkdt','linkenddt']].head(15))

(38216, 14)
     gvkey   datadate   permno     linkdt  linkenddt
0   001004 2020-05-31  54594.0 1972-04-24        NaT
1   001004 2021-05-31  54594.0 1972-04-24        NaT
2   001004 2022-05-31  54594.0 1972-04-24        NaT
3   001004 2023-05-31  54594.0 1972-04-24        NaT
4   001004 2024-05-31  54594.0 1972-04-24        NaT
5   001019 2020-12-31  10189.0 1972-12-14 1988-02-09
6   001045 2020-12-31  21020.0 1950-01-01 1962-01-30
7   001045 2020-12-31  21020.0 1962-01-31 2012-01-04
8   001045 2020-12-31  21020.0 2013-12-09        NaT
9   001045 2021-12-31  21020.0 1950-01-01 1962-01-30
10  001045 2021-12-31  21020.0 1962-01-31 2012-01-04
11  001045 2021-12-31  21020.0 2013-12-09        NaT
12  001045 2022-12-31  21020.0 1950-01-01 1962-01-30
13  001045 2022-12-31  21020.0 1962-01-31 2012-01-04
14  001045 2022-12-31  21020.0 2013-12-09        NaT


In [9]:
import pandas as pd
cond_start = comp_linked['datadate'] >= comp_linked['linkdt']
cond_end = (comp_linked['datadate'] <= comp_linked['linkenddt'])|comp_linked['linkenddt'].isna()

comp_linked = comp_linked[cond_start & cond_end].copy()

print("shape:", comp_linked.shape)
print(comp_linked[['gvkey', 'datadate', 'permno', 'linkdt', 'linkenddt']].head(15))

shape: (28676, 14)
     gvkey   datadate   permno     linkdt linkenddt
0   001004 2020-05-31  54594.0 1972-04-24       NaT
1   001004 2021-05-31  54594.0 1972-04-24       NaT
2   001004 2022-05-31  54594.0 1972-04-24       NaT
3   001004 2023-05-31  54594.0 1972-04-24       NaT
4   001004 2024-05-31  54594.0 1972-04-24       NaT
8   001045 2020-12-31  21020.0 2013-12-09       NaT
11  001045 2021-12-31  21020.0 2013-12-09       NaT
14  001045 2022-12-31  21020.0 2013-12-09       NaT
17  001045 2023-12-31  21020.0 2013-12-09       NaT
20  001045 2024-12-31  21020.0 2013-12-09       NaT
21  001050 2020-12-31  11499.0 1980-11-28       NaT
22  001050 2021-12-31  11499.0 1980-11-28       NaT
23  001050 2022-12-31  11499.0 1980-11-28       NaT
24  001050 2023-12-31  11499.0 1980-11-28       NaT
25  001050 2024-12-31  11499.0 1980-11-28       NaT


In [10]:
dup_count = comp_linked.duplicated(subset=['gvkey','datadate']).sum()
print("duplicate:", dup_count)

duplicate: 0


In [11]:
comp_linked['be']=comp_linked['ceq'].fillna(0)+comp_linked['txditc'].fillna(0)-comp_linked['pstk'].fillna(0)

print("be<=0: ", (comp_linked['be']<=0).sum())
print("total number of row: ",len(comp_linked))
print(comp_linked[['gvkey','datadate','permno','ceq','txditc','pstk','be']].head(10))

be<=0:  4197
total number of row:  28676
     gvkey   datadate   permno     ceq  txditc  pstk      be
0   001004 2020-05-31  54594.0   902.6     0.0   0.0   902.6
1   001004 2021-05-31  54594.0   974.4     9.5   0.0   983.9
2   001004 2022-05-31  54594.0  1034.5    20.0   0.0  1054.5
3   001004 2023-05-31  54594.0  1099.1    33.6   0.0  1132.7
4   001004 2024-05-31  54594.0  1189.8    23.9   0.0  1213.7
8   001045 2020-12-31  21020.0 -6867.0     9.0   0.0 -6858.0
11  001045 2021-12-31  21020.0 -7340.0     9.0   0.0 -7331.0
14  001045 2022-12-31  21020.0 -5799.0    10.0   0.0 -5789.0
17  001045 2023-12-31  21020.0 -5202.0     9.0   0.0 -5193.0
20  001045 2024-12-31  21020.0 -3977.0     9.0   0.0 -3968.0


In [12]:
# eliminate observations with BE <= 0
comp_clean = comp_linked[comp_linked['be']>0].copy()
print("shape afterwards:",comp_clean.shape)

# mark datadate as the fiscal year
comp_clean['fyear_end'] = comp_clean['datadate'].dt.year

print(comp_clean[['gvkey', 'datadate', 'permno', 'be', 'fyear_end']].head(10))

shape afterwards: (24479, 15)
     gvkey   datadate   permno       be  fyear_end
0   001004 2020-05-31  54594.0    902.6       2020
1   001004 2021-05-31  54594.0    983.9       2021
2   001004 2022-05-31  54594.0   1054.5       2022
3   001004 2023-05-31  54594.0   1132.7       2023
4   001004 2024-05-31  54594.0   1213.7       2024
21  001050 2020-12-31  11499.0  209.628       2020
22  001050 2021-12-31  11499.0  212.944       2021
23  001050 2022-12-31  11499.0   221.89       2022
24  001050 2023-12-31  11499.0  241.481       2023
25  001050 2024-12-31  11499.0  259.011       2024


In [13]:
# be effective date, datadate postponed by 6 months
comp_clean['be_available'] = comp_clean['datadate'] + pd.DateOffset(months=6)

print(comp_clean[['gvkey','permno','datadate','be','be_available']].head(10))

     gvkey   permno   datadate       be be_available
0   001004  54594.0 2020-05-31    902.6   2020-11-30
1   001004  54594.0 2021-05-31    983.9   2021-11-30
2   001004  54594.0 2022-05-31   1054.5   2022-11-30
3   001004  54594.0 2023-05-31   1132.7   2023-11-30
4   001004  54594.0 2024-05-31   1213.7   2024-11-30
21  001050  11499.0 2020-12-31  209.628   2021-06-30
22  001050  11499.0 2021-12-31  212.944   2022-06-30
23  001050  11499.0 2022-12-31   221.89   2023-06-30
24  001050  11499.0 2023-12-31  241.481   2024-06-30
25  001050  11499.0 2024-12-31  259.011   2025-06-30


In [16]:
# sort values on date
df_sorted = df.sort_values('date').copy()
comp_sorted = comp_clean.sort_values('be_available').copy()

df_sorted['permno'] = df_sorted['permno'].astype('Int64')
comp_sorted['permno'] = comp_sorted['permno'].astype('Int64')

# match crsp with be, be_available <= date
merged = pd.merge_asof(
    df_sorted,
    comp_sorted[['permno', 'be_available', 'be']],
    left_on = 'date',
    right_on = 'be_available',
    by = 'permno',
    direction = 'backward'
)

print(merged[['permno', 'date', 'ret', 'mktcap', 'be', 'be_available']].head(15))
print(merged[['permno', 'date', 'ret', 'mktcap', 'be', 'be_available']].tail(15))
print("shape:", merged.shape)

    permno       date       ret      mktcap    be be_available
0    10026 2020-01-31 -0.100016  3137526.96  <NA>          NaT
1    17623 2020-01-31 -0.035591    97947.75  <NA>          NaT
2    17622 2020-01-31   0.00589   32543.925  <NA>          NaT
3    12068 2020-01-31 -0.020776  3027757.14  <NA>          NaT
4    17621 2020-01-31  0.008275    108805.0  <NA>          NaT
5    17620 2020-01-31 -0.030425     43198.7  <NA>          NaT
6    91612 2020-01-31 -0.082321  690986.828  <NA>          NaT
7    17615 2020-01-31  -0.02667     5793.75  <NA>          NaT
8    17614 2020-01-31   0.02042     71775.0  <NA>          NaT
9    17613 2020-01-31 -0.017493   12661.695  <NA>          NaT
10   91614 2020-01-31 -0.000625   629324.36  <NA>          NaT
11   17610 2020-01-31       0.0    125792.0  <NA>          NaT
12   91615 2020-01-31  -0.07143      6370.0  <NA>          NaT
13   17608 2020-01-31  0.016633     83274.4  <NA>          NaT
14   17607 2020-01-31  0.006671    131970.0  <NA>      

In [17]:
print("total rows of na:", merged['be'].notna().sum())
print("total rows:", len(merged))

one = merged[merged['permno'] == 10026].sort_values('date')
print(one[['date', 'mktcap', 'be', 'be_available']].iloc[[0, 6, 12, 18, 24, 30]])

total rows of na: 212027
total rows: 546882
             date         mktcap       be be_available
0      2020-01-31     3137526.96     <NA>          NaT
50628  2020-07-31     2326541.35     <NA>          NaT
96859  2021-01-29      2897486.8     <NA>          NaT
147943 2021-07-30     3133740.32  873.911   2021-03-30
207947 2022-01-31      2898795.9  873.911   2021-03-30
259298 2022-07-29  2600707.72808  907.232   2022-03-30


In [18]:
# B/M
merged['bm'] = merged['be'] / merged['mktcap']

print(merged[['permno', 'date', 'mktcap', 'be', 'bm']].dropna(subset = ['bm']).head(10))
print("row number of bm:", merged['bm'].notna().sum())

       permno       date       mktcap        be        bm
46263   34948 2020-07-31    717870.92   545.138  0.000759
46358   17382 2020-07-31    1162836.0   166.228  0.000143
46451   83011 2020-07-31   6774364.08  1238.615  0.000183
46618   88360 2020-07-31  24435519.99  8709.813  0.000356
46765   80432 2020-07-31    1655000.0  1247.853  0.000754
47026   14141 2020-07-31   4648357.62    1417.0  0.000305
47075   14609 2020-07-31    701379.19    110.83  0.000158
47220   14296 2020-07-31     53884.87    130.78  0.002427
47398   14544 2020-07-31    281046.48   406.033  0.001445
47437   14803 2020-07-31   3963520.32  1160.787  0.000293
row number of bm: 210789
